# Constructing Documents for RAG

This tutorial is a continuation of the [Preparing Quran Dataset for RAG](https://aviosit.com/tutorials/preparing-quran-dataset-for-rag) tutorial, where we prepared a structured Quran dataset for downstream retrieval tasks.

In this tutorial, we transform the prepared dataset into documents suitable for a Retrieval-Augmented Generation (RAG) pipeline.

The same source data is represented in three ways:

1. **AviosIT Document** — framework-independent Python implementation.
2. **LangChain Document** — LangChain's native document representation.
3. **LlamaIndex Document** — LlamaIndex's native document representation.

Each document contains an identifier, text, and metadata. Keeping these concepts explicit makes documents easier to identify, filter, retrieve, and cite later in a RAG pipeline.

> **Note:** The example `Parquet` file contains synthetic placeholder text for demonstration only. It is not actual Quranic text or translations.

## Learning Objectives

By the end of this tutorial, you will be able to:

- Load a prepared `quran.parquet` dataset.
- Construct documents from individual ayah records in multiple languages.
- Separate document text from metadata.
- Understand the difference between a framework-independent document and framework-native document classes.
- Inspect the resulting document representations.

> **Prerequisite:** Complete the [Preparing Quran Dataset for RAG](https://aviosit.com/tutorials/preparing-quran-dataset-for-rag) tutorial, or use the example dataset provided in this repository.

## 1. Load the Prepared Dataset

The dataset contains one row per ayah and separate text columns for Arabic, English, and Indonesian.

In [1]:
from pathlib import Path

import pandas as pd


candidate_paths = [
    Path("constructing-documents-for-rag/data/processed/quran.parquet"),
    Path("../data/processed/quran.parquet"),
]

dataset_path = next((path for path in candidate_paths if path.exists()), None)

if dataset_path is None:
    raise FileNotFoundError(
        "Could not find quran.parquet. Run this notebook from the repository root "
        "or from constructing-documents-for-rag/notebooks/."
    )

quran_df = pd.read_parquet(dataset_path)

print(f"Loaded: {dataset_path}")
print(f"Rows: {len(quran_df):,}")
print(f"Columns: {quran_df.columns.tolist()}")

display(quran_df)

Loaded: ../data/processed/quran.parquet
Rows: 6,236
Columns: ['surah', 'ayah', 'arabic', 'english', 'indonesian']


,surah,ayah,arabic,english,indonesian
0,1,1,بِسْمِ ٱللَّهِ ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ,"In the name of Allah, the Entirely Merciful, t...",Dengan menyebut nama Allah Yang Maha Pemurah l...
1,1,2,ٱلْحَمْدُ لِلَّهِ رَبِّ ٱلْعَـٰلَمِينَ,"[All] praise is [due] to Allah, Lord of the wo...","Segala puji bagi Allah, Tuhan semesta alam."
2,1,3,ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ,"The Entirely Merciful, the Especially Merciful,",Maha Pemurah lagi Maha Penyayang.
3,1,4,مَـٰلِكِ يَوْمِ ٱلدِّينِ,Sovereign of the Day of Recompense.,Yang menguasai di Hari Pembalasan.
4,1,5,إِيَّاكَ نَعْبُدُ وَإِيَّاكَ نَسْتَعِينُ,It is You we worship and You we ask for help.,"Hanya Engkaulah yang kami sembah, dan hanya ke..."
...,...,...,...,...,...
6231,114,2,مَلِكِ ٱلنَّاسِ,The Sovereign of mankind.,Raja manusia.
6232,114,3,إِلَـٰهِ ٱلنَّاسِ,"The God of mankind,",Sembahan manusia.
6233,114,4,مِن شَرِّ ٱلْوَسْوَاسِ ٱلْخَنَّاسِ,From the evil of the retreating whisperer -,Dari kejahatan (bisikan) syaitan yang biasa be...
6234,114,5,ٱلَّذِى يُوَسْوِسُ فِى صُدُورِ ٱلنَّاسِ,Who whispers [evil] into the breasts of mankind -,yang membisikkan (kejahatan) ke dalam dada man...


## 2. Construct Documents

### 2.1 AviosIT Document

The framework-independent `Document` is a small data class containing the core RAG information:

- `id`
- `text`
- `metadata`

In [2]:
from dataclasses import dataclass
from typing import Any


@dataclass
class Document:
    id: str
    text: str
    metadata: dict[str, Any]


language_columns = {
    "ar": "arabic",
    "en": "english",
    "id": "indonesian",
}

aviosit_documents: list[Document] = []

for row in quran_df.itertuples(index=False):
    for language, column in language_columns.items():
        aviosit_documents.append(
            Document(
                id=f"{row.surah}:{row.ayah}:{language}",
                text=getattr(row, column),
                metadata={
                    "ayah_id": f"{row.surah}:{row.ayah}",
                    "surah": int(row.surah),
                    "ayah": int(row.ayah),
                    "language": language,
                },
            )
        )

print(f"Constructed {len(aviosit_documents):,} AviosIT documents.")

Constructed 18,708 AviosIT documents.


In [3]:
expected_count = len(quran_df) * len(language_columns)

assert len(aviosit_documents) == expected_count

print(f"Expected : {expected_count:,}")
print(f"Actual   : {len(aviosit_documents):,}")
print("Validation passed.")

Expected : 18,708
Actual   : 18,708
Validation passed.


### 2.2 LangChain Document

LangChain has its own native `Document` class.

- `id`
- `page_content`
- `metadata`
- `type`

In [4]:
from langchain_core.documents import Document as LangChainDocument


langchain_documents = [
    LangChainDocument(
        id=document.id,
        page_content=document.text,
        metadata=document.metadata.copy(),
    )
    for document in aviosit_documents
]

print(f"Constructed {len(langchain_documents):,} LangChain documents.")

Constructed 18,708 LangChain documents.


In [5]:
expected_count = len(quran_df) * len(language_columns)

assert len(langchain_documents) == expected_count

print(f"Expected : {expected_count:,}")
print(f"Actual   : {len(langchain_documents):,}")
print("Validation passed.")

Expected : 18,708
Actual   : 18,708
Validation passed.


### 2.3 LlamaIndex Document

LlamaIndex has its own native `Document` class.

Its model contains fields for metadata, relationships, embeddings, and separate text, image, audio, and video resources.

For this text-only RAG example, we populate the text content and metadata.

In [6]:
from llama_index.core import Document as LlamaIndexDocument


llamaindex_documents = [
    LlamaIndexDocument(
        id_=document.id,
        text=document.text,
        metadata=document.metadata.copy(),
    )
    for document in aviosit_documents
]

print(f"Constructed {len(llamaindex_documents):,} LlamaIndex documents.")

Constructed 18,708 LlamaIndex documents.


In [7]:
expected_count = len(quran_df) * len(language_columns)

assert len(llamaindex_documents) == expected_count

print(f"Expected : {expected_count:,}")
print(f"Actual   : {len(llamaindex_documents):,}")
print("Validation passed.")

Expected : 18,708
Actual   : 18,708
Validation passed.


## 3. Inspect the Document Objects

There are two useful ways to inspect a document:

1. **Print the object fields** to see the fields defined by the underlying object.
2. **Inspect the object** to see its normal representation.

These are not always the same thing. For example, LlamaIndex exposes `text` as a property while its model contains `text_resource`.

### 3.1 AviosIT Document

Print the object fields and its object.

In [8]:
print("\nFields:", aviosit_documents[0].__dataclass_fields__.keys())
aviosit_documents[0]


Fields: dict_keys(['id', 'text', 'metadata'])


Document(id='1:1:ar', text='بِسْمِ ٱللَّهِ ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ', metadata={'ayah_id': '1:1', 'surah': 1, 'ayah': 1, 'language': 'ar'})

The main values can also be accessed directly.

In [9]:
print(f'{aviosit_documents[0].id       = }')
print(f'{aviosit_documents[0].text     =}')
print(f'{aviosit_documents[0].metadata = }')

aviosit_documents[0].id       = '1:1:ar'
aviosit_documents[0].text     ='بِسْمِ ٱللَّهِ ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ'
aviosit_documents[0].metadata = {'ayah_id': '1:1', 'surah': 1, 'ayah': 1, 'language': 'ar'}


### 3.2 LangChain Document

Print the object fields and its object.

In [10]:
print("\nFields:", langchain_documents[0].model_dump().keys())
langchain_documents[0]


Fields: dict_keys(['id', 'metadata', 'page_content', 'type'])


Document(id='1:1:ar', metadata={'ayah_id': '1:1', 'surah': 1, 'ayah': 1, 'language': 'ar'}, page_content='بِسْمِ ٱللَّهِ ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ')

The main values can also be accessed directly.

In [11]:
print(f'{langchain_documents[0].id           = }')
print(f'{langchain_documents[0].page_content = }')
print(f'{langchain_documents[0].metadata     = }')

langchain_documents[0].id           = '1:1:ar'
langchain_documents[0].page_content = 'بِسْمِ ٱللَّهِ ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ'
langchain_documents[0].metadata     = {'ayah_id': '1:1', 'surah': 1, 'ayah': 1, 'language': 'ar'}


### 3.3 LlamaIndex Document

Print the object fields and its object.

Notice that `text` is not one of those model fields. It is a property that exposes the text resource.

In [12]:
print("\nFields:", llamaindex_documents[0].model_dump().keys())
llamaindex_documents[0]


Fields: dict_keys(['id_', 'embedding', 'metadata', 'excluded_embed_metadata_keys', 'excluded_llm_metadata_keys', 'relationships', 'metadata_template', 'metadata_separator', 'text_resource', 'image_resource', 'audio_resource', 'video_resource', 'text_template', 'class_name', 'text'])


Document(id_='1:1:ar', embedding=None, metadata={'ayah_id': '1:1', 'surah': 1, 'ayah': 1, 'language': 'ar'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='بِسْمِ ٱللَّهِ ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ', path=None, url=None, mimetype=None), image_resource=None, audio_resource=None, video_resource=None, text_template='{metadata_str}\n\n{content}')

The main values can also be accessed directly.

In [13]:
print(f'{llamaindex_documents[0].id_                 = }')
print(f'{llamaindex_documents[0].text                = }')
print(f'{llamaindex_documents[0].text_resource.text  = }')
print(f'{llamaindex_documents[0].metadata            = }')

llamaindex_documents[0].id_                 = '1:1:ar'
llamaindex_documents[0].text                = 'بِسْمِ ٱللَّهِ ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ'
llamaindex_documents[0].text_resource.text  = 'بِسْمِ ٱللَّهِ ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ'
llamaindex_documents[0].metadata            = {'ayah_id': '1:1', 'surah': 1, 'ayah': 1, 'language': 'ar'}


## 4. Compare the Three Document Representations

| Implementation | Text | ID | Metadata |
|---|---|---|---|
| AviosIT | `text` | `id` | `metadata` |
| LangChain | `page_content` | `id` | `metadata` |
| LlamaIndex | `text` property or `text_resource` | `id_` | `metadata` |

The three implementations do not need to have identical internal structures. The framework-independent implementation defines our own simple contract, while LangChain and LlamaIndex provide their own native document models.

For the inspection step, we print the actual document objects directly so they can be compared fairly.

## 5. Summary

We have now:

- loaded the prepared dataset;
- created AviosIT documents;
- created native LangChain documents;
- created native LlamaIndex documents;
- inspected their fields and document structure.

The documents are ready for the next RAG stages: **embedding, indexing, and retrieval**.